# Fine-tune a small open-source LLM on ANY document: PDF, DOCX, TXT, MD (Unsloth + QLoRA)

A self-contained Colab notebook (free **T4 GPU**). Nothing in it is specific to one document. It runs these steps:

| step | what happens |
|---|---|
| 1. Ingest | Format detected from file content: PDF (`pypdf`), DOCX (`python-docx`, including tables), TXT/MD. Then a sliding-window chunker (400 words, 50-word overlap) that keeps page numbers and chunk IDs |
| 2. Synthesize | A **teacher** LLM (`Qwen2.5-1.5B`) writes 3-5 grounded Q&A pairs per chunk (facts, definitions, procedures), each with 2 paraphrases. All prompts run in batches. |
| 3. Validate | Drops pairs whose numbers or named entities don't appear in the source chunk, and pairs whose answers aren't grounded in it (hallucination filter) |
| 4. Refusals | The teacher also writes questions the document can't answer. Any that resemble the document are dropped. The target answer is *"The provided document does not contain this information."* |
| 5. Split | 85% train / 15% test, **split by fact**, so test facts are never trained on |
| 6. Train | The **smallest student** (`Qwen2.5-0.5B`) is fine-tuned with QLoRA. Loss is computed only on answer tokens (prompt labels = `-100`). |
| 7. Evaluate | Token F1, ROUGE-L, key-fact recall, refusal accuracy for: **Base** vs **Fine-tuned** vs **Grounded in-prompt answering** (retrieved chunks in the prompt) |
| 8. Export | 16-bit merged model (vLLM), **GGUF q4_k_m + q8_0**, Ollama `Modelfile`, and local `llama-cpp-python` inference |

**Run:** *Runtime -> Change runtime type -> T4 GPU*. Upload your document (PDF, DOCX, TXT or MD) to `/content` and set `DOC_PATH` (if no file is found, a demo PDF is generated). Then *Run all*.

> **What to expect:** fine-tuning teaches the *style* and the facts it has seen. For facts the model has **never** been trained on (the test split), only grounded answering can know the answer, and the comparison table makes this visible. The best production setup is **fine-tuned model + retrieved context**.

## 1. Install

In [ ]:
!pip install -q unsloth pypdf python-docx scikit-learn reportlab

## 2. Configuration

In [24]:
DOC_PATH        = "/content/Sifayath_Ahmed_Resume_V6_Designed.docx"     # <- .pdf, .docx, .txt or .md (a demo PDF is created if missing)

STUDENT_MODEL   = "unsloth/Qwen2.5-0.5B-Instruct"  # model that is fine-tuned + exported (smallest)
TEACHER_MODEL   = "unsloth/Qwen2.5-1.5B-Instruct"  # writes the synthetic Q&A ("unsloth/Qwen2.5-3B-Instruct" = better data, slower)
USE_UNSLOTH     = True                              # False -> plain transformers + peft + bitsandbytes
MAX_SEQ_LENGTH  = 4096
LOAD_IN_4BIT    = True

# Chunking
CHUNK_WORDS, CHUNK_OVERLAP = 400, 50
MAX_CHUNKS      = 80          # cap for speed; raise for long PDFs (each chunk ~ a few seconds of generation)

# Synthesis
QA_PER_CHUNK    = 4           # 3-5 Q&A pairs per chunk
N_PARAPHRASES   = 2
N_REFUSAL       = 40          # out-of-scope questions to generate
GEN_BATCH_SIZE  = 8           # prompts generated in parallel

# Split / retrieval / training
TEST_FRACTION   = 0.15
TOP_K           = 2           # retrieved chunks for grounded answering
CONTEXT_FRACTION = 0.3        # share of training examples that include retrieved context (teaches grounded use)
EPOCHS          = 3
LEARNING_RATE   = 2e-4
LORA_R, LORA_ALPHA = 16, 32
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
MAX_EXAMPLE_TOKENS = 1536
MAX_STEPS       = -1          # e.g. 20 for a smoke test
EVAL_MAX_ITEMS  = 60          # per evaluation split (speed)
SEED            = 3407

REFUSAL = "The provided document does not contain this information."
OUTPUT_NAME = "doc-assistant"

## 3. Ingest the document (PDF / DOCX / TXT / MD): extraction and sliding-window chunking

In [25]:
import os, re, json, random, math, gc
from pypdf import PdfReader

def make_demo_pdf(path):
    # Small fictional manual, used only when no PDF was uploaded.
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, PageBreak
    ss = getSampleStyleSheet()
    sections = [
        ("Overview", "The AquaPure X3 is a countertop water purifier made by Lumen Home Appliances. It uses a "
         "four-stage filtration system and purifies up to 2 litres of water per minute. The warranty period is 24 months."),
        ("Filters", "The sediment filter must be replaced every 6 months. The carbon block filter lasts 12 months. "
         "The RO membrane should be replaced every 24 months. Genuine filters have the part number prefix LH-F."),
        ("Error codes", "Error E1 means low water pressure: check that the inlet valve is fully open. Error E4 means the "
         "filter life has expired: replace the filter and hold the RESET button for 5 seconds. Error E7 indicates a leak: "
         "unplug the unit immediately and contact support."),
        ("Installation", "Install the unit within 1.5 metres of a power outlet. The inlet water pressure must be between "
         "1 and 4 bar. Flush the system for 10 minutes before first use."),
        ("Support", "Customer support is available from 8 AM to 8 PM, Monday to Saturday, at 1-800-555-0142. "
         "Service visits within the warranty period are free of charge."),
        ("Cleaning", "Clean the storage tank every 3 months with a mild food-grade cleaner. Never use bleach, because "
         "it damages the RO membrane. Wipe the exterior with a dry cloth only."),
        ("Water quality", "The TDS indicator shows the total dissolved solids in parts per million (ppm). A reading below "
         "50 ppm is shown in green, 50 to 150 ppm in amber, and above 150 ppm in red. A red reading means the RO "
         "membrane must be checked by a technician."),
        ("Power and safety", "The unit uses 24 watts in operation and 1 watt in standby. It must be connected to a "
         "grounded 220-240 V outlet. Do not install the purifier outdoors or in direct sunlight."),
        ("Warranty terms", "The 24-month warranty covers manufacturing defects but does not cover filters, which are "
         "consumables. The warranty is void if non-genuine filters are used. Register the product within 30 days "
         "of purchase to receive 6 extra months of warranty."),
        ("Troubleshooting", "If the water tastes bad, replace the carbon block filter. If the flow is slow, check the "
         "sediment filter first and then the inlet pressure. If the unit makes a clicking noise every few minutes, "
         "this is the normal sound of the auto-flush valve, which runs every 4 hours."),
    ]
    sections = sections * 3        # long enough for several 400-word chunks
    story = [Paragraph("AquaPure X3 User and Service Manual", ss["Title"])]
    for i, (h, body) in enumerate(sections):
        story += [Paragraph(h, ss["Heading2"]), Paragraph(body, ss["BodyText"]), Spacer(1, 10)]
        if i % 4 == 3: story.append(PageBreak())
    SimpleDocTemplate(path, pagesize=A4, title="AquaPure X3 User and Service Manual").build(story)

if not os.path.exists(DOC_PATH):
    DOC_PATH = "demo_manual.pdf"; make_demo_pdf(DOC_PATH)
    print("No document found -> using demo PDF:", DOC_PATH)

WORDS_PER_PAGE = 500     # DOCX/TXT have no fixed pages -> approximate "pages" of ~500 words for metadata

def detect_format(path):
    # Decide by file CONTENT, not extension (a renamed .docx still works).
    with open(path, "rb") as fh:
        head = fh.read(8)
    if head.startswith(b"%PDF"):
        return "pdf"
    if head.startswith(b"PK\x03\x04"):
        import zipfile
        names = zipfile.ZipFile(path).namelist()
        if "word/document.xml" in names:
            return "docx"
        raise ValueError(f"{path} is a ZIP-based Office file but not .docx (e.g. .pptx/.xlsx) - export it to PDF first.")
    if head.startswith(b"\xd0\xcf\x11\xe0"):
        raise ValueError(f"{path} is a legacy .doc file - save it as .docx or PDF first.")
    return "text"

def _clean(t):
    t = t.replace("\u00ad", "")
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)            # de-hyphenate line breaks
    return re.sub(r"[ \t]+", " ", t).strip()

def _paginate(blocks):
    # group paragraphs into ~WORDS_PER_PAGE-word pseudo-pages
    pages, cur, n = [], [], 0
    for b in blocks:
        cur.append(b); n += len(b.split())
        if n >= WORDS_PER_PAGE:
            pages.append("\n".join(cur)); cur, n = [], 0
    if cur:
        pages.append("\n".join(cur))
    return pages

def _docx_blocks(path):
    # paragraphs and tables in document order (nested tables included, merged cells de-duplicated)
    from docx import Document
    from docx.table import Table
    from docx.text.paragraph import Paragraph
    doc = Document(path)
    def walk(parent, element):
        for child in element.iterchildren():
            tag = child.tag.split("}")[-1]
            if tag == "p":
                yield Paragraph(child, parent).text
            elif tag == "tbl":
                seen = set()
                for row in Table(child, parent).rows:
                    cells = []
                    for cell in row.cells:
                        if cell._tc in seen:
                            continue
                        seen.add(cell._tc)
                        cells.append([t for t in walk(cell, cell._tc) if t.strip()])
                    if all(len(c) <= 1 for c in cells):          # data table -> one line per row
                        yield " | ".join(c[0] for c in cells if c)
                    else:                                         # layout table (e.g. designed resume) -> its paragraphs
                        for c in cells:
                            yield from c
    blocks = [_clean(t) for t in walk(doc, doc.element.body)]
    return [b for b in blocks if b], (doc.core_properties.title or "").strip()

def _default_title(path, title):
    if not title or title.lower() in {"(anonymous)", "untitled"} or title.lower().startswith("microsoft word"):
        title = os.path.splitext(os.path.basename(path))[0].replace("_", " ")
    return title

def extract_pages(path):
    fmt = detect_format(path)
    if fmt == "pdf":
        reader = PdfReader(path)
        pages = [_clean(p.extract_text() or "") for p in reader.pages]
        title = (reader.metadata.title or "") if reader.metadata else ""
    elif fmt == "docx":
        blocks, title = _docx_blocks(path)
        pages = _paginate(blocks)
    else:
        raw = open(path, "rb").read().decode("utf-8", errors="replace")
        pages, title = _paginate([_clean(b) for b in re.split(r"\n\s*\n", raw) if b.strip()]), ""
    return pages, _default_title(path, title.strip()), fmt

def chunk_pages(pages, size=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    # Sliding word window across the whole document; each chunk keeps the pages it spans.
    words = [(w, i + 1) for i, p in enumerate(pages) for w in p.split()]
    chunks, step = [], max(1, size - overlap)
    for start in range(0, len(words), step):
        win = words[start:start + size]
        if len(win) < 30 and chunks:                       # tiny tail: merge into previous chunk
            break
        chunks.append({"chunk_id": f"c{len(chunks):04d}", "page_start": win[0][1], "page_end": win[-1][1],
                       "text": " ".join(w for w, _ in win)})
        if start + size >= len(words):
            break
    return chunks

pages, DOC_TITLE, DOC_FORMAT = extract_pages(DOC_PATH)
n_words = sum(len(p.split()) for p in pages)
assert n_words > 50, "Almost no text extracted - a scanned PDF needs OCR first (e.g. ocrmypdf)."
chunks = chunk_pages(pages)
print(f"'{DOC_TITLE}' [{DOC_FORMAT}]: {len(pages)} {'pages' if DOC_FORMAT == 'pdf' else '~500-word sections'}, "
      f"{n_words} words -> {len(chunks)} chunks")
if len(chunks) > MAX_CHUNKS:
    print(f"Using the first {MAX_CHUNKS} chunks (raise MAX_CHUNKS for full coverage)")
    chunks = chunks[:MAX_CHUNKS]
print(json.dumps(chunks[0], indent=1)[:600])

No document found -> using demo PDF: demo_manual.pdf
'AquaPure X3 User and Service Manual' [pdf]: 8 pages, 1140 words -> 4 chunks
{
 "chunk_id": "c0000",
 "page_start": 1,
 "page_end": 3,
 "text": "AquaPure X3 User and Service Manual Overview The AquaPure X3 is a countertop water purifier made by Lumen Home Appliances. It uses a four-stage filtration system and purifies up to 2 litres of water per minute. The warranty period is 24 months. Filters The sediment filter must be replaced every 6 months. The carbon block filter lasts 12 months. The RO membrane should be replaced every 24 months. Genuine filters have the part number prefix LH-F. Error codes Error E1 means low water pressure: check that the inlet valve is fully 


### Retriever (TF-IDF) for grounded answering
A lightweight retriever with no extra model downloads. It is used for grounded evaluation, for context-augmented training examples, and to filter refusal questions.

In [26]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english").fit([c["text"] for c in chunks])
chunk_matrix = tfidf.transform([c["text"] for c in chunks])

def retrieve(query, k=TOP_K):
    sims = (chunk_matrix @ tfidf.transform([query]).T).toarray().ravel()
    idx = np.argsort(-sims)[:k]
    return [(chunks[i], float(sims[i])) for i in idx]

def build_context(query, k=TOP_K):
    return "\n\n---\n\n".join(f"[pages {c['page_start']}-{c['page_end']}] {c['text']}" for c, _ in retrieve(query, k))

# top keywords describe the document's topic (used to generate out-of-scope questions)
weights = np.asarray(chunk_matrix.sum(axis=0)).ravel()
TOPIC_KEYWORDS = [tfidf.get_feature_names_out()[i] for i in np.argsort(-weights)[:20]]
print("Topic keywords:", ", ".join(TOPIC_KEYWORDS))

Topic keywords: warranty, filter, filters, water, months, unit, 24, inlet, pressure, error, ppm, sediment filter, sediment, minutes, check, carbon, block, carbon block, replace, membrane


## 4. Model helpers (loading, batched generation)

In [27]:
import torch
if USE_UNSLOTH:
    from unsloth import FastLanguageModel          # import before transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(name):
    if USE_UNSLOTH:
        m, tok = FastLanguageModel.from_pretrained(model_name=name, max_seq_length=MAX_SEQ_LENGTH,
                                                   dtype=None, load_in_4bit=LOAD_IN_4BIT)
    else:
        kw = {}
        if LOAD_IN_4BIT and torch.cuda.is_available():
            from transformers import BitsAndBytesConfig
            kw = {"quantization_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                            bnb_4bit_use_double_quant=True,
                                                            bnb_4bit_compute_dtype=torch.float16),
                  "device_map": "auto"}
        tok = AutoTokenizer.from_pretrained(name)
        m = AutoModelForCausalLM.from_pretrained(name, **kw)
        if not kw:
            m.to("cuda" if torch.cuda.is_available() else "cpu")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return m, tok

@torch.no_grad()
def generate_batch(m, tok, conversations, max_new_tokens=256, batch_size=GEN_BATCH_SIZE):
    # Greedy, batched generation. Returns only the newly generated text for each conversation.
    if USE_UNSLOTH:
        FastLanguageModel.for_inference(m)
    m.eval()
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(conversations), batch_size):
        prompts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
                   for c in conversations[i:i + batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(m.device)
        out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05,
                         pad_token_id=tok.pad_token_id, use_cache=True)
        outs += [tok.decode(r, skip_special_tokens=True).strip() for r in out[:, enc["input_ids"].shape[1]:]]
    return outs

def free(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. Synthetic Q&A generation (teacher model)
For every chunk, the teacher writes grounded Q&A pairs of several types, each with paraphrases. All chunks are processed in batches.

In [28]:
QA_PROMPT = '''You create training data from a document excerpt.
Write {n} diverse question-answer pairs that are answered ONLY by the excerpt below.
Mix the types: factual recall (names, numbers, dates), definitions, and procedural steps ("how do I...").
Rules:
- Each answer must be fully supported by the excerpt. Never use outside knowledge.
- Answers: complete but concise (1-4 sentences). Keep numbers, units and names exactly as written.
- Questions must make sense on their own (mention the subject, not "the excerpt").
- For each question write {p} paraphrases with different wording and the same meaning.
Return ONLY a JSON list:
[{{"type": "factual|definition|procedural", "question": "...", "paraphrases": ["...", "..."], "answer": "..."}}]

EXCERPT (document: {title}, pages {ps}-{pe}):
<excerpt>
{text}
</excerpt>'''

def parse_json_items(raw):
    # Robust JSON extraction: whole list first, then object by object (small models often break JSON).
    m = re.search(r"\[.*\]", raw, re.S)
    if m:
        try:
            data = json.loads(m.group(0))
            if isinstance(data, list):
                return [d for d in data if isinstance(d, dict)]
        except json.JSONDecodeError:
            pass
    items = []
    for obj in re.findall(r"\{[^{}]*\}", raw, re.S):
        try:
            items.append(json.loads(obj))
        except json.JSONDecodeError:
            continue
    return items

teacher, teacher_tok = load_model(TEACHER_MODEL)
prompts = [[{"role": "user", "content": QA_PROMPT.format(n=QA_PER_CHUNK, p=N_PARAPHRASES, title=DOC_TITLE,
                                                          ps=c["page_start"], pe=c["page_end"], text=c["text"])}]
           for c in chunks]
raw_outputs = generate_batch(teacher, teacher_tok, prompts, max_new_tokens=900)

raw_pairs = []
for c, raw in zip(chunks, raw_outputs):
    for it in parse_json_items(raw):
        q, a = str(it.get("question", "")).strip(), str(it.get("answer", "")).strip()
        if q and a:
            paras = [str(p).strip() for p in it.get("paraphrases", []) if isinstance(p, str) and p.strip()]
            raw_pairs.append({"chunk_id": c["chunk_id"], "type": it.get("type", "factual"),
                              "question": q, "paraphrases": paras[:N_PARAPHRASES], "answer": a})
print(f"Teacher produced {len(raw_pairs)} raw Q&A pairs from {len(chunks)} chunks")

==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=900) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Teacher produced 16 raw Q&A pairs from 4 chunks


### Hallucination validation
A pair is kept only if:
* every **number** in the answer appears in the source chunk,
* every **named entity / acronym** in the answer (capitalised terms, ignoring sentence-initial words) appears in the chunk,
* at least 60% of the answer's content words occur in the chunk (the answer is grounded),
* the question and answer have sensible lengths, and the question is not a duplicate.

In [29]:
STOP = set('''a an the and or but of to in on for with by at from as is are was were be been being this that these
those it its they them their there here you your we our i if then than when what which who whom how why where do does
did can could should would will may might must not no yes also into onto over under about after before such each any
all some more most other only own same so too very just'''.split())

def norm(s):
    s = s.lower().replace("’", "'").replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", s)

def numbers(text):
    return {n.rstrip(".,") for n in re.findall(r"\d[\d.,:/%-]*", text)}

def entities(text):
    # lower-case sentence-initial capitalised words so they are not mistaken for named entities
    t = re.sub(r"(^|[.!?:]\s+|\n)([A-Z])(?=[a-z])", lambda m: m.group(1) + m.group(2).lower(), text)
    ents = set()
    for m in re.finditer(r"\b[A-Z][\w&-]*(?:\s+[A-Z][\w&-]*)*\b", t):
        e = m.group(0).strip()
        if e.lower() not in STOP and len(e) > 1:
            ents.add(e)
    return ents

def content_words(text):
    return [w for w in re.findall(r"[a-z0-9]+", norm(text)) if len(w) > 3 and w not in STOP]

def key_facts(text):
    return {norm(x) for x in numbers(text) | entities(text)}

def validate(pair, chunk_text):
    src = norm(chunk_text)
    a, q = pair["answer"], pair["question"]
    if not (3 <= len(q.split()) <= 40 and 2 <= len(a.split()) <= 150):
        return False, "length"
    if any(n not in src for n in numbers(a)):
        return False, "number not in source"
    if any(norm(e) not in src for e in entities(a)):
        return False, "entity not in source"
    cw = content_words(a)
    if cw and sum(w in src for w in cw) / len(cw) < 0.6:
        return False, "answer not grounded"
    return True, "ok"

chunk_by_id = {c["chunk_id"]: c for c in chunks}
facts, rejects, seen_q, by_answer = [], {}, set(), {}
for p in raw_pairs:
    ok, why = validate(p, chunk_by_id[p["chunk_id"]]["text"])
    if ok and norm(p["question"]) in seen_q:
        ok, why = False, "duplicate"
    if not ok:
        rejects[why] = rejects.get(why, 0) + 1
        continue
    new_qs = [q for q in [p["question"]] + p["paraphrases"] if norm(q) not in seen_q and not seen_q.add(norm(q))]
    akey = norm(p["answer"])
    if akey in by_answer:            # same fact generated twice (repeated text) -> one group, so it can't leak across the split
        by_answer[akey]["paraphrases"] += new_qs
        rejects["merged duplicate fact"] = rejects.get("merged duplicate fact", 0) + 1
        continue
    p["question"], p["paraphrases"] = new_qs[0], new_qs[1:]
    by_answer[akey] = p
    facts.append(p)

print(f"Kept {len(facts)} / {len(raw_pairs)} pairs. Rejected: {rejects}")
assert len(facts) >= 4, "Too few valid pairs - try TEACHER_MODEL='unsloth/Qwen2.5-3B-Instruct' or a text-based PDF"
for f in facts[:3]:
    print(json.dumps(f, ensure_ascii=False, indent=1)[:500])

Kept 11 / 16 pairs. Rejected: {'answer not grounded': 1, 'merged duplicate fact': 3, 'duplicate': 1}
{
 "chunk_id": "c0000",
 "type": "factual",
 "question": "What is the maximum water output rate of the AquaPure X3?",
 "paraphrases": [
  "How much water can the AquaPure X3 produce per minute?",
  "What is the highest water output rate of the AquaPure X3?"
 ],
 "answer": "The AquaPure X3 purifies up to 2 litres of water per minute."
}
{
 "chunk_id": "c0000",
 "type": "factual",
 "question": "How often should the RO membrane be replaced?",
 "paraphrases": [
  "When should the RO membrane be changed?",
  "At what frequency should the RO membrane be replaced?"
 ],
 "answer": "The RO membrane should be replaced every 24 months."
}
{
 "chunk_id": "c0000",
 "type": "procedural",
 "question": "What is the recommended installation distance from the power outlet?",
 "paraphrases": [
  "What is the optimal distance from the power outlet?",
  "How far away from the power outlet should the AquaPur

### Out-of-scope / refusal dataset (generated, then filtered)
The teacher writes questions the document does **not** answer, given only the document's topic keywords. Any question that resembles the document (TF-IDF similarity to a chunk) is dropped. If too few remain, a small built-in list fills the gap.

In [ ]:
REFUSAL_PROMPT = '''A document is about: {kw}.
Write {n} diverse questions that this document would certainly NOT answer: general world knowledge, trivia,
history, science, sports, cooking, other professions and unrelated products. One question per line, no numbering.'''

raw = generate_batch(teacher, teacher_tok, [[{"role": "user", "content": REFUSAL_PROMPT.format(
    kw=", ".join(TOPIC_KEYWORDS), n=N_REFUSAL)}]], max_new_tokens=900)[0]
cands = [re.sub(r"^\s*(?:\d+[.)]|[-*•])\s*", "", l).strip() for l in raw.splitlines()]
cands = [q for q in cands if q.endswith("?") and 4 <= len(q.split()) <= 25]

FALLBACK = ["What is the capital of Australia?", "Who painted the Mona Lisa?", "How many players are on a football team?",
            "What is the boiling point of water in Fahrenheit?", "Who wrote Romeo and Juliet?",
            "How do I bake sourdough bread?", "What is the speed of light?", "Which planet is the largest in the solar system?",
            "What year did World War II end?", "How do I change a car tyre?", "What is photosynthesis?",
            "Who was the first person to walk on the moon?", "What is the tallest mountain in Africa?",
            "How many bones are in the human body?", "What is the chemical symbol for gold?", "Translate 'thank you' into French."]

def out_of_scope(q, max_sim=0.12):
    return retrieve(q, 1)[0][1] < max_sim

refusal_qs = list(dict.fromkeys(q for q in cands if out_of_scope(q)))
if len(refusal_qs) < N_REFUSAL // 2:
    refusal_qs += [q for q in FALLBACK if out_of_scope(q) and q not in refusal_qs]
refusal_qs = refusal_qs[:N_REFUSAL]
print(f"{len(refusal_qs)} refusal questions, e.g.:", refusal_qs[:5])

free(teacher); del teacher, teacher_tok; free()

## 6. Train / test split (by fact, 85 / 15)
* **test_unseen**: 15% of facts (question + paraphrases), **never trained on**. This checks generalisation, which mostly requires grounding.
* **test_paraphrase**: one held-out paraphrase of each *trained* fact. This checks whether fine-tuning really learned the fact (not just the wording).
* **refusal**: out-of-scope questions are split 85 / 15 as well.

In [30]:
rng = random.Random(SEED)
rng.shuffle(facts)
n_test = max(1, round(len(facts) * TEST_FRACTION))
test_facts, train_facts = facts[:n_test], facts[n_test:]

train_qa, test_paraphrase = [], []
for f in train_facts:
    qs = [f["question"]] + f["paraphrases"]
    if len(qs) >= 2:
        test_paraphrase.append({"question": qs[-1], "answer": f["answer"], "chunk_id": f["chunk_id"]})
        qs = qs[:-1]
    train_qa += [{"question": q, "answer": f["answer"], "chunk_id": f["chunk_id"]} for q in qs]
test_unseen = [{"question": f["question"], "answer": f["answer"], "chunk_id": f["chunk_id"]} for f in test_facts]

rq = refusal_qs[:]; rng.shuffle(rq)
n_rt = max(1, round(len(rq) * TEST_FRACTION))
test_refusal = [{"question": q, "answer": REFUSAL} for q in rq[:n_rt]]
train_refusal = [{"question": q, "answer": REFUSAL} for q in rq[n_rt:]]

# leakage check: no test question/answer pair appears in training
train_keys = {norm(r["question"]) for r in train_qa}
assert not any(norm(r["question"]) in train_keys for r in test_unseen + test_paraphrase)

rng.shuffle(test_paraphrase)
test_paraphrase = test_paraphrase[:EVAL_MAX_ITEMS]
test_unseen = test_unseen[:EVAL_MAX_ITEMS]
os.makedirs("dataset", exist_ok=True)
for name, rows in {"train_qa": train_qa, "train_refusal": train_refusal, "test_unseen": test_unseen,
                   "test_paraphrase": test_paraphrase, "test_refusal": test_refusal}.items():
    with open(f"dataset/{name}.jsonl", "w") as fh:
        fh.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
    print(f"{name:16s} {len(rows)}")

train_qa         27
train_refusal    14
test_unseen      2
test_paraphrase  9
test_refusal     2


## 7. Evaluation framework
* **Token F1** and **ROUGE-L**: word overlap with the reference answer (ROUGE-L also rewards correct order).
* **Fact recall**: share of the reference's key facts (numbers, named entities) that appear in the answer.
* **Refusal accuracy**: out-of-scope questions answered with the refusal sentence.
* **False refusal rate**: in-document questions that were wrongly refused.

In [31]:
from collections import Counter
import pandas as pd

SYSTEM_PROMPT = (f"You are a helpful assistant for the document '{DOC_TITLE}'. Answer questions accurately and "
                 f"concisely using only the document. If the document does not contain the answer, reply exactly: "
                 f"'{REFUSAL}'")

def build_messages(question, context=None):
    system = SYSTEM_PROMPT
    if context:
        system += "\n\nRelevant document excerpts:\n" + context
    return [{"role": "system", "content": system}, {"role": "user", "content": question}]

def tokens(s): return re.findall(r"[a-z0-9]+", norm(s))

def token_f1(pred, ref):
    p, r = tokens(pred), tokens(ref)
    common = sum((Counter(p) & Counter(r)).values())
    if not p or not r or not common: return 0.0
    pr, rc = common / len(p), common / len(r)
    return 2 * pr * rc / (pr + rc)

def rouge_l(pred, ref):
    p, r = tokens(pred), tokens(ref)
    if not p or not r: return 0.0
    prev = [0] * (len(r) + 1)
    for pw in p:
        cur = [0]
        for j, rw in enumerate(r):
            cur.append(prev[j] + 1 if pw == rw else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]
    if not l: return 0.0
    pr, rc = l / len(p), l / len(r)
    return 2 * pr * rc / (pr + rc)

def fact_recall(pred, ref):
    kf = key_facts(ref)
    if not kf: return None
    p = norm(pred)
    return sum(f in p for f in kf) / len(kf)

def is_refusal(text):
    t = norm(text)
    return "does not contain this information" in t or bool(re.search(
        r"(doesn't|does not|don't|do not) (contain|mention|provide|have|cover)|no information|not (mentioned|provided|covered)", t))

EVAL_ROWS = []
def evaluate(label, m, tok, grounded=False):
    def run(rows):
        msgs = [build_messages(r["question"], build_context(r["question"]) if grounded else None) for r in rows]
        return generate_batch(m, tok, msgs, max_new_tokens=160)
    res = {"system": label}
    for split, rows in [("unseen", test_unseen), ("paraphrase", test_paraphrase)]:
        if not rows: continue
        preds = run(rows)
        fr = [x for x in (fact_recall(p, r["answer"]) for p, r in zip(preds, rows)) if x is not None]
        res[f"{split}_F1"] = np.mean([token_f1(p, r["answer"]) for p, r in zip(preds, rows)])
        res[f"{split}_RougeL"] = np.mean([rouge_l(p, r["answer"]) for p, r in zip(preds, rows)])
        res[f"{split}_FactRecall"] = np.mean(fr) if fr else np.nan
        res[f"{split}_FalseRefusal"] = np.mean([is_refusal(p) for p in preds])
        for p, r in list(zip(preds, rows))[:2]:
            print(f"  [{split}] Q: {r['question']}\n     ref : {r['answer'][:150]}\n     pred: {p[:150]}")
    preds = run(test_refusal)
    res["Refusal_Acc"] = np.mean([is_refusal(p) for p in preds])
    EVAL_ROWS.append(res)
    print(f"==> {label}: " + ", ".join(f"{k}={v:.2f}" for k, v in res.items() if k != "system"))
    return res

def show_results():
    return pd.DataFrame(EVAL_ROWS).set_index("system").round(3)

## 8. Load the student (smallest model) and measure baselines

In [32]:
model, tokenizer = load_model(STUDENT_MODEL)
evaluate("1. Base (no context)", model, tokenizer)
evaluate("2. Base + grounded (retrieved context)", model, tokenizer, grounded=True)
show_results()

==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [unseen] Q: What does the 24-month warranty cover?
     ref : The 24-month warranty covers manufacturing defects but does not cover filters, which are consumables.
     pred: The 24-month warranty covers the following items:

1. The product itself
2. Any accessories or attachments that are attached to the product
3. The pro
  [unseen] Q: How often does the auto-flush valve run according to the troubleshooting instructions?
     ref : The auto-flush valve runs every 4 hours.
     pred: The auto-flush valve runs approximately once every 10 minutes according to the troubleshooting instructions.


Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [unseen] Q: What does the 24-month warranty cover?
     ref : The 24-month warranty covers manufacturing defects but does not cover filters, which are consumables.
     pred: The 24-month warranty covers manufacturing defects but does not cover filters, which are consumables. The warranty is void if non-genuine filters are 
  [unseen] Q: How often does the auto-flush valve run according to the troubleshooting instructions?
     ref : The auto-flush valve runs every 4 hours.
     pred: Every 4 hours.


Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [paraphrase] Q: At what frequency should the sediment filter be changed?
     ref : The sediment filter must be replaced every 6 months.
     pred: Every 6 months.
  [paraphrase] Q: What is the highest water output rate of the AquaPure X3?
     ref : The AquaPure X3 purifies up to 2 litres of water per minute.
     pred: The highest water output rate of the AquaPure X3 is 2 litres per minute.
==> 2. Base + grounded (retrieved context): unseen_F1=0.65, unseen_RougeL=0.65, unseen_FactRecall=1.00, unseen_FalseRefusal=0.50, paraphrase_F1=0.62, paraphrase_RougeL=0.57, paraphrase_FactRecall=1.00, paraphrase_FalseRefusal=0.00, Refusal_Acc=0.00


,unseen_F1,unseen_RougeL,unseen_FactRecall,unseen_FalseRefusal,paraphrase_F1,paraphrase_RougeL,paraphrase_FactRecall,paraphrase_FalseRefusal,Refusal_Acc
system,,,,,,,,,
1. Base (no context),0.363,0.363,0.5,0.0,0.403,0.368,0.381,0.0,0.0
2. Base + grounded (retrieved context),0.648,0.648,1.0,0.5,0.619,0.574,1.000,0.0,0.0


## 9. Tokenise with answer-only loss
Prompt tokens (system + optional context + question) get label `-100`, so only the answer, including the closing
end-of-turn token, contributes to the loss. Some examples include retrieved context, so the model also learns to use excerpts and to refuse when they don't help.

In [33]:
from datasets import Dataset

def encode(row):
    msgs = build_messages(row["question"], row.get("context"))
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(msgs + [{"role": "assistant", "content": row["answer"]}], tokenize=False)
    assert full.startswith(prompt)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids), "labels": [-100] * len(p_ids) + f_ids[len(p_ids):]}

train_rows = []
for r in train_qa + train_refusal:
    row = dict(r)
    if rng.random() < CONTEXT_FRACTION:
        row["context"] = build_context(r["question"])
    train_rows.append({"question": row["question"], "answer": row["answer"], "context": row.get("context")})
rng.shuffle(train_rows)

train_ds = (Dataset.from_list(train_rows).map(encode, remove_columns=["question", "answer", "context"])
            .filter(lambda ex: len(ex["input_ids"]) <= MAX_EXAMPLE_TOKENS))   # drop, never truncate (keeps EOS)
ex = train_ds[0]
print(f"{len(train_ds)} training examples")
print("Loss is computed on:", repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))

Map:   0%|          | 0/41 [00:00<?, ? examples/s]

Filter:   0%|          | 0/41 [00:00<?, ? examples/s]

41 training examples
Loss is computed on: 'The auto-flush valve runs every 4 hours and is responsible for flushing the system automatically.<|im_end|>\n'


## 10. QLoRA fine-tuning (Unsloth)

In [34]:
import inspect
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

if USE_UNSLOTH:
    model = FastLanguageModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0,
                                             target_modules=TARGET_MODULES, bias="none",
                                             use_gradient_checkpointing="unsloth", random_state=SEED)
    FastLanguageModel.for_training(model)
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if LOAD_IN_4BIT and torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                                             target_modules=TARGET_MODULES, task_type="CAUSAL_LM"))
model.print_trainable_parameters()
tokenizer.padding_side = "right"

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
grouping = ({"group_by_length": True} if "group_by_length" in inspect.signature(TrainingArguments.__init__).parameters
            else {"train_sampling_strategy": "group_by_length"})          # name differs across transformers versions
args = TrainingArguments(
    output_dir="outputs", num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_steps=5, weight_decay=0.0,
    fp16=torch.cuda.is_available() and not bf16, bf16=bf16,
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    logging_steps=10, save_strategy="no", report_to="none", seed=SEED, **grouping,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
trainer.train()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 41 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss


TrainOutput(global_step=9, training_loss=0.657860173119439, metrics={'train_runtime': 47.0561, 'train_samples_per_second': 2.614, 'train_steps_per_second': 0.191, 'total_flos': 198159024364800.0, 'train_loss': 0.657860173119439, 'epoch': 3.0})

## 11. Evaluate: Base vs Fine-tuned vs Grounded

In [35]:
evaluate("3. Fine-tuned (no context)", model, tokenizer)
evaluate("4. Fine-tuned + grounded", model, tokenizer, grounded=True)
results = show_results()
results.to_csv("evaluation_results.csv")
results

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [unseen] Q: What does the 24-month warranty cover?
     ref : The 24-month warranty covers manufacturing defects but does not cover filters, which are consumables.
     pred: The provided document does not contain this information.
  [unseen] Q: How often does the auto-flush valve run according to the troubleshooting instructions?
     ref : The auto-flush valve runs every 4 hours.
     pred: The auto-flush valve runs every 12 months according to the troubleshooting instructions.


Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [paraphrase] Q: At what frequency should the sediment filter be changed?
     ref : The sediment filter must be replaced every 6 months.
     pred: The sediment filter must be replaced every 6 months.
  [paraphrase] Q: What is the highest water output rate of the AquaPure X3?
     ref : The AquaPure X3 purifies up to 2 litres of water per minute.
     pred: The AquaPure X3 has a maximum output rate of 100 litres per hour.


Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [unseen] Q: What does the 24-month warranty cover?
     ref : The 24-month warranty covers manufacturing defects but does not cover filters, which are consumables.
     pred: The 24-month warranty covers manufacturing defects but does not cover filters, which are consumables.
  [unseen] Q: How often does the auto-flush valve run according to the troubleshooting instructions?
     ref : The auto-flush valve runs every 4 hours.
     pred: The unit makes a clicking noise every few minutes, indicating that the auto-flush valve runs every 4 hours.


Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=160) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [paraphrase] Q: At what frequency should the sediment filter be changed?
     ref : The sediment filter must be replaced every 6 months.
     pred: The sediment filter must be replaced every 6 months.
  [paraphrase] Q: What is the highest water output rate of the AquaPure X3?
     ref : The AquaPure X3 purifies up to 2 litres of water per minute.
     pred: The AquaPure X3 purifies up to 2 litres of water per minute.
==> 4. Fine-tuned + grounded: unseen_F1=0.80, unseen_RougeL=0.80, unseen_FactRecall=1.00, unseen_FalseRefusal=0.50, paraphrase_F1=0.85, paraphrase_RougeL=0.84, paraphrase_FactRecall=1.00, paraphrase_FalseRefusal=0.00, Refusal_Acc=1.00


,unseen_F1,unseen_RougeL,unseen_FactRecall,unseen_FalseRefusal,paraphrase_F1,paraphrase_RougeL,paraphrase_FactRecall,paraphrase_FalseRefusal,Refusal_Acc
system,,,,,,,,,
1. Base (no context),0.363,0.363,0.5,0.0,0.403,0.368,0.381,0.000,0.0
2. Base + grounded (retrieved context),0.648,0.648,1.0,0.5,0.619,0.574,1.000,0.000,0.0
3. Fine-tuned (no context),0.416,0.416,0.0,0.5,0.612,0.591,0.524,0.222,1.0
4. Fine-tuned + grounded,0.796,0.796,1.0,0.5,0.854,0.840,1.000,0.000,1.0


**How to read the table**
* `paraphrase_*`: trained facts asked in new words. The fine-tuned model should beat the base model clearly here.
* `unseen_*`: facts never trained on. Only grounded rows can do well, which is expected.
* `Refusal_Acc`: should rise sharply after fine-tuning. `*_FalseRefusal` should stay low.
* For production, use **4. Fine-tuned + grounded** (retrieve chunks, then ask the fine-tuned model).

## 12. Save the adapter and the 16-bit merged model (vLLM / Transformers)

In [36]:
model.save_pretrained("lora_adapter"); tokenizer.save_pretrained("lora_adapter")
MERGED_DIR = "merged_model"
if USE_UNSLOTH:
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
else:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL, dtype=torch.float16)
    merged = PeftModel.from_pretrained(base, "lora_adapter").merge_and_unload()
    merged.save_pretrained(MERGED_DIR); tokenizer.save_pretrained(MERGED_DIR)
    del base, merged; gc.collect()
with open(f"{MERGED_DIR}/system_prompt.txt", "w") as fh:
    fh.write(SYSTEM_PROMPT)
print(sorted(os.listdir(MERGED_DIR)))
# vLLM:  vllm serve merged_model --dtype half --max-model-len 4096    (fresh runtime; T4 has no bf16)

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 1509.29it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:18<00:00, 18.08s/it]


Unsloth: Merge process complete. Saved to `/content/merged_model`
['.cache', 'chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'system_prompt.txt', 'tokenizer.json', 'tokenizer_config.json']


## 13. GGUF export (q4_k_m + q8_0)
Unsloth's exporter runs first. If it fails, the cell falls back to llama.cpp: it converts `merged_model/` to q8_0, and builds `llama-quantize` (about 2-4 min) to make q4_k_m.

In [37]:
import glob, subprocess, shutil
GGUF_QUANTS = ["q4_k_m", "q8_0"]
gguf_files = {}

def _match(paths, q):
    q = q.replace("_", "").lower()
    hits = [p for p in paths if q in os.path.basename(p).replace("_", "").lower()]
    return max(hits, key=os.path.getmtime) if hits else None

if USE_UNSLOTH:
    try:
        model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method=GGUF_QUANTS)
        found = glob.glob("gguf_model*/**/*.gguf", recursive=True) + glob.glob("*.gguf")
        for q in GGUF_QUANTS:
            src = _match(found, q)
            if src:
                gguf_files[q] = f"{OUTPUT_NAME}-{q}.gguf"; os.replace(src, gguf_files[q])
    except Exception as e:
        print("Unsloth GGUF export failed -> llama.cpp fallback:", e)

missing = [q for q in GGUF_QUANTS if q not in gguf_files]
if missing:
    run = lambda cmd: subprocess.run(cmd, check=True)
    if not os.path.exists("llama.cpp"):
        run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp"])
    run(["pip", "install", "-q", "gguf", "sentencepiece"])
    convert = ["python", "llama.cpp/convert_hf_to_gguf.py", MERGED_DIR]
    if "q8_0" in missing:
        run(convert + ["--outfile", f"{OUTPUT_NAME}-q8_0.gguf", "--outtype", "q8_0"])
        gguf_files["q8_0"] = f"{OUTPUT_NAME}-q8_0.gguf"
    if "q4_k_m" in missing:
        try:
            run(convert + ["--outfile", f"{OUTPUT_NAME}-f16.gguf", "--outtype", "f16"])
            run(["cmake", "-S", "llama.cpp", "-B", "llama.cpp/build", "-DLLAMA_CURL=OFF", "-DGGML_NATIVE=OFF"])
            run(["cmake", "--build", "llama.cpp/build", "--target", "llama-quantize", "-j", str(os.cpu_count())])
            qbin = glob.glob("llama.cpp/build/bin/llama-quantize*")[0]
            run([qbin, f"{OUTPUT_NAME}-f16.gguf", f"{OUTPUT_NAME}-q4_k_m.gguf", "Q4_K_M"])
            gguf_files["q4_k_m"] = f"{OUTPUT_NAME}-q4_k_m.gguf"
            os.remove(f"{OUTPUT_NAME}-f16.gguf")
        except Exception as e:
            print("q4_k_m quantization skipped:", e)

for q, f in gguf_files.items():
    print(f"{q:7s} -> {f}  ({os.path.getsize(f) / 1e6:.0f} MB)")
GGUF_PATH = gguf_files.get("q4_k_m") or gguf_files.get("q8_0")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 3241.35it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:07<00:00,  7.02s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m', 'q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['gguf_model_gguf/Qwen2.5-0.5B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['gguf_model_gguf/Qw

## 14. Ollama Modelfile

In [38]:
with open("Modelfile", "w") as fh:
    fh.write(f"FROM ./{os.path.basename(GGUF_PATH)}\n"
             f"PARAMETER temperature 0\nPARAMETER num_ctx {MAX_SEQ_LENGTH}\n"
             f'SYSTEM \"\"\"{SYSTEM_PROMPT}\"\"\"\n')
print(open("Modelfile").read())
print(f"Usage:  ollama create {OUTPUT_NAME} -f Modelfile   &&   ollama run {OUTPUT_NAME}")

FROM ./doc-assistant-q4_k_m.gguf
PARAMETER temperature 0
PARAMETER num_ctx 4096
SYSTEM """You are a helpful assistant for the document 'AquaPure X3 User and Service Manual'. Answer questions accurately and concisely using only the document. If the document does not contain the answer, reply exactly: 'The provided document does not contain this information.'"""

Usage:  ollama create doc-assistant -f Modelfile   &&   ollama run doc-assistant


## 15. Local inference with `llama-cpp-python` (CPU, same code works on a laptop)
The chat template is stored inside the GGUF file. Pass retrieved context for the most accurate answers.

In [ ]:
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [39]:
from llama_cpp import Llama
llm = Llama(model_path=GGUF_PATH, n_ctx=MAX_SEQ_LENGTH, n_threads=os.cpu_count(), verbose=False)

def ask_gguf(question, grounded=True, max_tokens=200):
    ctx = build_context(question) if grounded else None
    out = llm.create_chat_completion(messages=build_messages(question, ctx), temperature=0.0, max_tokens=max_tokens)
    return out["choices"][0]["message"]["content"].strip()

for r in (test_paraphrase[:2] + test_unseen[:1] + test_refusal[:1]):
    print("Q   :", r["question"])
    print("ref :", r["answer"][:200])
    print("GGUF:", ask_gguf(r["question"]), "\n")

Q   : At what frequency should the sediment filter be changed?
ref : The sediment filter must be replaced every 6 months.
GGUF: The sediment filter must be replaced every 6 months. 

Q   : What is the highest water output rate of the AquaPure X3?
ref : The AquaPure X3 purifies up to 2 litres of water per minute.
GGUF: The AquaPure X3 purifies up to 2 litres of water per minute. 

Q   : What does the 24-month warranty cover?
ref : The 24-month warranty covers manufacturing defects but does not cover filters, which are consumables.
GGUF: The warranty covers manufacturing defects but does not cover filters, which are consumables. 

Q   : What is the chemical symbol for gold?
ref : The provided document does not contain this information.
GGUF: The provided document does not contain this information. 



### Standalone use (laptop, no notebook)
```python
from llama_cpp import Llama
llm = Llama(model_path="doc-assistant-q4_k_m.gguf", n_ctx=4096, verbose=False)
system = open("system_prompt.txt").read()          # copied from merged_model/
r = llm.create_chat_completion(messages=[{"role": "system", "content": system},
                                         {"role": "user", "content": "Your question here"}],
                               temperature=0, max_tokens=200)
print(r["choices"][0]["message"]["content"])
```

## 16. Download the artifacts

In [ ]:
bundle = f"{OUTPUT_NAME}_artifacts"
os.makedirs(bundle, exist_ok=True)
for f in list(gguf_files.values()) + ["Modelfile", "evaluation_results.csv", f"{MERGED_DIR}/system_prompt.txt"]:
    shutil.copy(f, bundle)
shutil.copytree("dataset", f"{bundle}/dataset", dirs_exist_ok=True)
shutil.make_archive(bundle, "zip", bundle)
print(f"{bundle}.zip", f"{os.path.getsize(bundle + '.zip') / 1e6:.0f} MB")
try:
    from google.colab import files
    files.download(f"{bundle}.zip")
except ImportError:
    print("Not in Colab - artifacts are in", os.getcwd())

In [40]:
from llama_cpp import Llama
import os

# 1. Load your exported GGUF model (update the filename to match your export)
GGUF_PATH = "doc-assistant-q4_k_m.gguf"
llm = Llama(model_path=GGUF_PATH, n_ctx=4096, n_threads=os.cpu_count(), verbose=False)

# 2. Define the exact system prompt used during training
SYSTEM_PROMPT = "You are an assistant that answers questions about Sifayath's resume accurately and concisely. If the resume does not contain the answer, reply: 'The resume does not contain that information.'"

# 3. Define a clean inference function
def ask_resume(question, context_text=None):
    sys_content = SYSTEM_PROMPT

    # If you want grounded answering, inject the raw document text here
    if context_text:
        sys_content += f"\n\nAnswer ONLY from this resume:\n<resume>\n{context_text}\n</resume>"

    messages = [
        {"role": "system", "content": sys_content},
        {"role": "user", "content": question}
    ]

    out = llm.create_chat_completion(
        messages=messages,
        temperature=0.0,
        max_tokens=200
    )
    return out["choices"][0]["message"]["content"].strip()

# 4. Direct questions tailored to your resume
my_questions = [
    "What is Sifayath's current role at HCLTech?",
    "Can you summarize the candidate's experience with LangGraph and multi-agent systems?",
    "What role did Sifayath play in the AI Foundry Agentic Hub project?",
    "Does the resume mention experience with FastAPI, Python, and RAG pipelines?",
    "How many years of professional enterprise architecture experience does Sifayath have?"
]

print("Asking the fine-tuned model directly...\n" + "-"*50)

# Optional: If you have your resume loaded as a string, pass it as context_text=doc_text
for q in my_questions:
    print(f"Q: {q}")
    print(f"A: {ask_resume(q)}\n")

Asking the fine-tuned model directly...
--------------------------------------------------
Q: What is Sifayath's current role at HCLTech?
A: The resume does not contain that information.

Q: Can you summarize the candidate's experience with LangGraph and multi-agent systems?
A: The candidate has experience with LangGraph and multi-agent systems.

Q: What role did Sifayath play in the AI Foundry Agentic Hub project?
A: The resume does not contain that information.

Q: Does the resume mention experience with FastAPI, Python, and RAG pipelines?
A: The resume does not contain that information.

Q: How many years of professional enterprise architecture experience does Sifayath have?
A: The resume does not contain that information.

